In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * SCHEMA DESIGN: From Flat Tables to Production-Ready Models
 * ─────────────────────────────────────────────────────────────────────
*/

In [ ]:
/**
 * TRANSFORMING RAW DATA INTO STRUCTURED, INTEGRITY-PROTECTED DATABASES
 * ─────────────────────────────────────────────────────────────────────
 *
 * TABLE OF CONTENTS:
 * 1. THE MINDSET SHIFT ................. Convenience → Correctness
 * 2. IDENTIFY CORE ENTITIES ............ Students, Instructors, Courses
 * 3. STUDENTS TABLE .................... Surrogate keys vs. natural keys
 * 4. STUDENT PROFILES TABLE ............ Sparse data separation
 * 5. INSTRUCTORS TABLE ................. Same principles as students
 * 6. COURSES TABLE ..................... Constraints and referential integrity
 * 7. ENROLLMENTS (BRIDGE TABLE) ........ Many-to-many relationships
 * 8. PAYMENTS TABLE .................... Financial integrity
 * ─────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 1. THE MINDSET SHIFT — Convenience → Correctness
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ WHEN YOU MOVE FROM SPREADSHEET TO PRODUCTION DATABASE:             │
 * │                                                                    │
 * │   The biggest shift is NOT syntax.                                 │
 * │   It is MINDSET.                                                   │
 * │                                                                    │
 * │   FROM: Designing for CONVENIENCE                                  │
 * │     → "How can I answer this query quickly?"                       │
 * │     → "Let's put everything in one table"                          │
 * │     → "I'll store the total as a column"                           │
 * │                                                                    │
 * │   TO: Designing for CORRECTNESS, INTEGRITY, and SCALE              │
 * │     → "What are the real-world entities?"                          │
 * │     → "How do they relate?"                                        │
 * │     → "How do I prevent corruption?"                               │
 * │     → "How does this scale?"                                       │
 * │                                                                    │
 * │   The result is not just "better tables."                          │
 * │   It's a database that:                                            │
 * │     → Enforces its own rules                                       │
 * │     → Protects itself from bad data                                │
 * │     → Scales without breaking                                      │
 * │     → Can be audited and trusted                                   │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 2. IDENTIFY CORE ENTITIES — What are the real-world objects?
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ FROM FIRST PRINCIPLES:                                             │
 * │                                                                    │
 * │   We identify the real-world objects in our domain:                │
 * │                                                                    │
 * │   ┌─────────────────────────────────────────────────────────┐      │
 * │   │ Entity          │ What it represents                    │      │
 * │   ├─────────────────┼───────────────────────────────────────┤      │
 * │   │ Student         │ A person who learns                   │      │
 * │   │ Student Profile │ Extended details (optional)           │      │
 * │   │ Instructor      │ A person who teaches                  │      │
 * │   │ Course          │ A learning offering                   │      │
 * │   │ Enrollment      │ A student joining a course            │      │
 * │   │ Payment         │ A financial transaction               │      │
 * │   └─────────────────┴───────────────────────────────────────┘      │
 * │                                                                    │
 * │   Each entity gets its OWN TABLE.                                  │
 * │   This is the foundation of relational design.                     │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 3. STUDENTS TABLE — Surrogate keys vs. natural keys
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE DESIGN:                                                        │
 * │                                                                    │
 * │   CREATE TABLE students (                                          │
 * │     student_id INT AUTO_INCREMENT PRIMARY KEY,                     │
 * │     email VARCHAR(255) NOT NULL,                                   │
 * │     full_name VARCHAR(100) NOT NULL,                               │
 * │     created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,        │
 * │     CONSTRAINT uq_students_email UNIQUE (email)                    │
 * │   );                                                               │
 * │                                                                    │
 * │ KEY DECISION: Surrogate Key (student_id)                           │
 * │                                                                    │
 * │   We generate our own internal ID.                                 │
 * │   We do NOT use email as primary key.                              │
 * │                                                                    │
 * │ WHY NOT USE EMAIL AS PRIMARY KEY?                                  │
 * │                                                                    │
 * │   Email is UNIQUE, yes. But email is NOT STABLE.                   │
 * │                                                                    │
 * │   If email is primary key:                                         │
 * │     → It becomes foreign key in:                                   │
 * │         • enrollments                                              │
 * │         • payments                                                 │
 * │         • audit logs                                               │
 * │         • activity tables                                          │
 * │     → Student changes email → must update EVERYWHERE!              │
 * │                                                                    │
 * │   This causes:                                                     │
 * │     → Update anomalies                                             │
 * │     → Risk of partial updates                                      │
 * │     → Lock contention                                              │
 * │     → Integrity bugs                                               │
 * │     → Foreign key cascade nightmares                               │
 * │                                                                    │
 * │   Surrogate key (student_id) is:                                   │
 * │     → Stable (never changes)                                       │
 * │     → Compact (INT vs VARCHAR)                                     │
 * │     → Efficient for indexes                                        │
 * │     → Safe for foreign keys                                        │
 * │                                                                    │
 * │   Email is UNIQUE, but it's an ATTRIBUTE, not the IDENTITY.        │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 4. STUDENT PROFILES TABLE — Sparse data separation
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE DESIGN:                                                        │
 * │                                                                    │
 * │   CREATE TABLE student_profiles (                                  │
 * │     student_id INT PRIMARY KEY,                                    │
 * │     country_code CHAR(2) NULL,                                     │
 * │     graduation_year INT NULL,                                      │
 * │     github_url VARCHAR(255) NULL,                                  │
 * │     FOREIGN KEY (student_id)                                       │
 * │       REFERENCES students(student_id)                              │
 * │       ON DELETE CASCADE                                            │
 * │       ON UPDATE RESTRICT                                           │
 * │   );                                                               │
 * │                                                                    │
 * │ WHY SEPARATE TABLE?                                                │
 * │                                                                    │
 * │   Not every student provides:                                      │
 * │     → GitHub URL                                                   │
 * │     → Graduation year                                              │
 * │     → Location                                                     │
 * │     → LinkedIn                                                     │
 * │                                                                    │
 * │   If stored in students table:                                     │
 * │     → Most rows would contain NULL                                 │
 * │     → Waste space                                                  │
 * │     → Waste cache memory                                           │
 * │     → Clutter core table                                           │
 * │                                                                    │
 * │   PRODUCTION PRINCIPLE:                                            │
 * │     → Keep core identity tables LEAN                               │
 * │     → Move optional/sparse data to separate tables                 │
 * │                                                                    │
 * │   This improves:                                                   │
 * │     → Read performance (smaller rows)                              │
 * │     → Cache efficiency (more rows per page)                        │
 * │     → Table clarity (core identity vs. extras)                     │
 * │                                                                    │
 * │ WHY ON DELETE CASCADE HERE?                                        │
 * │                                                                    │
 * │   Profile has no meaning without student.                          │
 * │   This is OWNERSHIP.                                               │
 * │   Ownership relationships use CASCADE.                             │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 5. INSTRUCTORS TABLE — Same principles as students
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE DESIGN:                                                        │
 * │                                                                    │
 * │   CREATE TABLE instructors (                                       │
 * │     instructor_id INT AUTO_INCREMENT PRIMARY KEY,                  │
 * │     email VARCHAR(255) NOT NULL,                                   │
 * │     full_name VARCHAR(100) NOT NULL,                               │
 * │     CONSTRAINT uq_instructors_email UNIQUE (email)                 │
 * │   );                                                               │
 * │                                                                    │
 * │ SAME LOGIC AS STUDENTS:                                            │
 * │   → Surrogate key (instructor_id) for stability                    │
 * │   → Email is UNIQUE but an attribute, not the key                  │
 * │   → Never tie relationships to mutable business attributes         │
 * │                                                                    │
 * │ WHY THIS MATTERS:                                                  │
 * │   If an instructor changes email:                                  │
 * │     → Update ONE row in instructors table                          │
 * │     → No foreign key updates needed                                │
 * │     → No cascade nightmares                                        │
 * │     → System remains consistent                                    │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 6. COURSES TABLE — Constraints and referential integrity
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌──────────────────────────────────────────────────────────────────────────┐
 * │ THE DESIGN:                                                              │
 * │                                                                          │
 * │   CREATE TABLE courses (                                                 │
 * │     course_id INT AUTO_INCREMENT PRIMARY KEY,                            │
 * │     course_code VARCHAR(30) NOT NULL,                                    │
 * │     title VARCHAR(200) NOT NULL,                                         │
 * │     instructor_id INT NOT NULL,                                          │
 * │     base_price DECIMAL(10,2) NOT NULL,                                   │
 * │     currency CHAR(3) NOT NULL,                                           │
 * │     is_active TINYINT NOT NULL DEFAULT 1,                                │
 * │                                                                          │
 * │     CONSTRAINT uq_courses_code UNIQUE (course_code),                     │
 * │     CONSTRAINT ck_courses_currency CHECK (currency IN ('INR','USD')),    │
 * │     CONSTRAINT ck_courses_price CHECK (base_price >= 0),                 │
 * │                                                                          │
 * │     CONSTRAINT fk_courses_instructor                                     │
 * │       FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)  │
 * │       ON DELETE RESTRICT                                                 │
 * │       ON UPDATE RESTRICT                                                 │
 * │   );                                                                     │
 * │                                                                          │
 * │ DATA INTEGRITY CONSTRAINTS:                                              │
 * │                                                                          │
 * │   → UNIQUE(course_code) → prevents duplicate course codes                │
 * │   → CHECK(currency IN (...)) → prevents invalid currency                 │
 * │   → CHECK(base_price >= 0) → prevents negative prices                    │
 * │                                                                          │
 * │   PRODUCTION RULE:                                                       │
 * │     Never trust application validation alone.                            │
 * │     Enforce correctness at the DATABASE LAYER.                           │
 * │                                                                          │
 * │ WHY RESTRICT ON DELETE?                                                  │
 * │                                                                          │
 * │   If an instructor teaches a course:                                     │
 * │     → You cannot delete that instructor                                  │
 * │                                                                          │
 * │   If you allowed deletion:                                               │
 * │     → Course would point to non-existent instructor                      │
 * │     → Frontend joins fail                                                │
 * │     → Reports break                                                      │
 * │     → Analytics corrupt                                                  │
 * │                                                                          │
 * │   RESTRICT protects referential integrity.                               │
 * │   Deletion must be intentional and safe.                                 │
 * └──────────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 7. ENROLLMENTS (BRIDGE TABLE) — Many-to-many relationships
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────────────────────┐
 * │ THE DESIGN:                                                                        │
 * │                                                                                    │
 * │   CREATE TABLE enrollments (                                                       │
 * │     enrollment_id INT AUTO_INCREMENT PRIMARY KEY,                                  │
 * │     student_id INT NOT NULL,                                                       │
 * │     course_id INT NOT NULL,                                                        │
 * │     enrolled_at DATETIME NOT NULL,                                                 │
 * │     status VARCHAR(20) NOT NULL DEFAULT 'ACTIVE',                                  │
 * │                                                                                    │
 * │     CONSTRAINT ck_enroll_status CHECK (status IN ('ACTIVE','COMPLETED','DROPPED')),│
 * │                                                                                    │
 * │     CONSTRAINT fk_enroll_student                                                   │
 * │       FOREIGN KEY (student_id) REFERENCES students(student_id)                     │
 * │       ON DELETE RESTRICT                                                           │
 * │       ON UPDATE RESTRICT,                                                          │
 * │                                                                                    │
 * │     CONSTRAINT fk_enroll_course                                                    │
 * │       FOREIGN KEY (course_id) REFERENCES courses(course_id)                        │
 * │       ON DELETE RESTRICT                                                           │
 * │       ON UPDATE RESTRICT,                                                          │
 * │                                                                                    │
 * │     -- Prevent duplicate enrollment for same student+course                        │
 * │     CONSTRAINT uq_enroll_unique UNIQUE (student_id, course_id)                     │
 * │   );                                                                               │
 * │                                                                                    │
 * │ WHY BRIDGE TABLE?                                                                  │
 * │                                                                                    │
 * │   Students ↔ Courses is MANY-TO-MANY:                                              │
 * │     → One student can take many courses                                            │
 * │     → One course can have many students                                            │
 * │                                                                                    │
 * │   WRONG DESIGN (denormalized):                                                     │
 * │     courses_enrolled = [1, 4, 7]  ← comma-separated list                           │
 * │     → Breaks First Normal Form                                                     │
 * │     → Cannot filter efficiently                                                    │
 * │     → Cannot index properly                                                        │
 * │     → Data consistency nightmare                                                   │
 * │                                                                                    │
 * │   CORRECT DESIGN:                                                                  │
 * │     enrollments table with one row per enrollment                                  │
 * │     Each row = ONE FACT (student joins course)                                     │
 * │     Bridge tables convert N:M into two 1:N relationships                           │
 * │                                                                                    │
 * │ WHY UNIQUE(student_id, course_id)?                                                 │
 * │                                                                                    │
 * │   Prevents duplicate enrollments.                                                  │
 * │   Without this:                                                                    │
 * │     → Student could enroll in same course 3 times                                  │
 * │     → Data corruption possible                                                     │
 * │     → Business rule violation                                                      │
 * │                                                                                    │
 * │   Constraints prevent SILENT CORRUPTION.                                           │
 * └────────────────────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 8. PAYMENTS TABLE — Financial integrity
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌──────────────────────────────────────────────────────────────────────────────┐
 * │ THE DESIGN:                                                                  │
 * │                                                                              │
 * │   CREATE TABLE payments (                                                    │
 * │     payment_pk INT AUTO_INCREMENT PRIMARY KEY,                               │
 * │     payment_id VARCHAR(40) NOT NULL,      -- external gateway ID             │
 * │     student_id INT NOT NULL,                                                 │
 * │     course_id INT NOT NULL,                                                  │
 * │     amount DECIMAL(10,2) NOT NULL,                                           │
 * │     currency CHAR(3) NOT NULL,                                               │
 * │     status VARCHAR(20) NOT NULL,                                             │
 * │     created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,                  │
 * │     paid_at DATETIME NULL,                                                   │
 * │                                                                              │
 * │     CONSTRAINT uq_payments_payment_id UNIQUE (payment_id),                   │
 * │     CONSTRAINT ck_pay_currency CHECK (currency IN ('INR','USD')),            │
 * │     CONSTRAINT ck_pay_amount CHECK (amount >= 0),                            │
 * │     CONSTRAINT ck_pay_status CHECK (status IN ('PAID','PENDING','REFUNDED')),│
 * │                                                                              │
 * │     CONSTRAINT fk_pay_student                                                │
 * │       FOREIGN KEY (student_id) REFERENCES students(student_id)               │
 * │       ON DELETE RESTRICT                                                     │
 * │       ON UPDATE RESTRICT,                                                    │
 * │                                                                              │
 * │     CONSTRAINT fk_pay_course                                                 │
 * │       FOREIGN KEY (course_id) REFERENCES courses(course_id)                  │
 * │       ON DELETE RESTRICT                                                     │
 * │       ON UPDATE RESTRICT                                                     │
 * │   );                                                                         │
 * │                                                                              │
 * │ WHY SEPARATE PAYMENT PRIMARY KEY?                                            │
 * │                                                                              │
 * │   payment_pk → internal database identifier                                  │
 * │   payment_id → external gateway reference (Stripe, Razorpay)                 │
 * │                                                                              │
 * │   Never depend on external ID as primary key:                                │
 * │     → External systems change formats                                        │
 * │     → IDs might not be guaranteed unique across gateways                     │
 * │     → You need your own stable identifier                                    │
 * │                                                                              │
 * │ WHY RESTRICT FOR PAYMENTS?                                                   │
 * │                                                                              │
 * │   Payments are FINANCIAL RECORDS.                                            │
 * │   You cannot:                                                                │
 * │     → Delete student with payment history                                    │
 * │     → Delete course with paid transactions                                   │
 * │                                                                              │
 * │   Financial integrity > convenience.                                         │
 * │   RESTRICT ensures auditability.                                             │
 * │                                                                              │
 * │ UNIQUE(payment_id):                                                          │
 * │   → Prevents duplicate payments from same gateway transaction                │
 * │   → Protects against accidental double processing                            │
 * └──────────────────────────────────────────────────────────────────────────────┘
 */